<a href="https://colab.research.google.com/github/DigoShane/C-/blob/main/Extract_Data_from_Colorbar_plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os

# Upload the PNG
uploaded = files.upload()
filename = next(iter(uploaded))

image = Image.open(filename).convert("RGB")
rgb = np.asarray(image)

image_height, image_width = rgb.shape[:2]

print("Filename:", filename)
print("Image width :", image_width)
print("Image height:", image_height)

# Display the image with pixel coordinates
plt.figure(figsize=(16, 10))
plt.imshow(rgb)

plt.xlim(0, image_width)
plt.ylim(image_height, 0)

plt.xlabel("Pixel x-coordinate")
plt.ylabel("Pixel y-coordinate")
plt.xticks(np.linspace(0, image_width, 11))
plt.yticks(np.linspace(0, image_height, 11))
plt.grid(alpha=0.3)

plt.show()

In [ ]:
import numpy as np

# ============================================================
# USER INPUT
# ============================================================

# Colored field boundaries:
#
# (x_left, x_right, y_top, y_bottom)
#
# The boundaries are inclusive.
FIELD_BOX = (
    347,    # x_left
    1258,   # x_right
    79,     # y_top
    990     # y_bottom
)

# Interior of the colorbar.
# Avoid including the black colorbar border.
#
# (x_left, x_right, y_top, y_bottom)
COLORBAR_BOX = (
    1490,   # x_left
    1525,   # x_right
    47,     # y_top
    1023    # y_bottom
)

# Use either:
#     "vertical"
#     "horizontal"
COLORBAR_ORIENTATION = "vertical"

# Pixel positions of visible colorbar ticks.
#
# For a vertical colorbar, enter y-pixel coordinates.
# For a horizontal colorbar, enter x-pixel coordinates.
TICK_PIXELS = np.array([
    67,
    238,
    409,
    580,
    750,
    921
], dtype=float)

# Numerical values corresponding to TICK_PIXELS
TICK_VALUES = np.array([
     4.0e-4,
     2.0e-4,
     0.0,
    -2.0e-4,
    -4.0e-4,
    -6.0e-4
], dtype=float)

# Physical coordinate range represented by the colored field
X_RANGE = (0.0, 1.0)
Y_RANGE = (0.0, 1.0)

# Colorbar scaling:
#
# "linear" for ordinary colorbars
# "log10" for logarithmic positive-valued colorbars
COLORBAR_SCALE = "linear"

# Pixels whose color is too different from every colorbar color
# are excluded. Increase this if too many field pixels are removed.
MAX_RGB_DISTANCE = 8.0

# Number of contour levels
NUMBER_OF_CONTOURS = 30

# Use:
#     "field"    -> levels span recovered field min/max
#     "colorbar" -> levels span the complete colorbar range
CONTOUR_RANGE = "field"

# Create a large long-format CSV containing x, y, value.
# This can be very large for high-resolution images.
SAVE_LONG_FORMAT_CSV = False

In [ ]:
from scipy.spatial import cKDTree
from matplotlib.colors import ListedColormap
from google.colab import files

import numpy as np
import matplotlib.pyplot as plt
import os
import shutil


def validate_box(box, image_width, image_height, name):
    """Check that an image bounding box is valid."""

    x_left, x_right, y_top, y_bottom = box

    if not (0 <= x_left < x_right < image_width):
        raise ValueError(
            f"{name}: x-coordinates must satisfy "
            f"0 <= x_left < x_right < {image_width}."
        )

    if not (0 <= y_top < y_bottom < image_height):
        raise ValueError(
            f"{name}: y-coordinates must satisfy "
            f"0 <= y_top < y_bottom < {image_height}."
        )


def calibrate_colorbar(pixel_coordinates, values, scale):
    """
    Construct a mapping from colorbar pixel coordinate to field value.
    """

    pixel_coordinates = np.asarray(pixel_coordinates, dtype=float)
    values = np.asarray(values, dtype=float)

    if len(pixel_coordinates) != len(values):
        raise ValueError(
            "TICK_PIXELS and TICK_VALUES must have the same length."
        )

    if len(pixel_coordinates) < 2:
        raise ValueError("At least two colorbar ticks are required.")

    if scale == "linear":

        slope, intercept = np.polyfit(
            pixel_coordinates,
            values,
            deg=1
        )

        def pixel_to_value(pixel):
            return slope * pixel + intercept

        calibration_description = (
            f"value = ({slope:.12e}) * pixel "
            f"+ ({intercept:.12e})"
        )

    elif scale == "log10":

        if np.any(values <= 0):
            raise ValueError(
                "All tick values must be positive for log10 scaling."
            )

        slope, intercept = np.polyfit(
            pixel_coordinates,
            np.log10(values),
            deg=1
        )

        def pixel_to_value(pixel):
            return 10.0 ** (slope * pixel + intercept)

        calibration_description = (
            f"log10(value) = ({slope:.12e}) * pixel "
            f"+ ({intercept:.12e})"
        )

    else:
        raise ValueError(
            "COLORBAR_SCALE must be 'linear' or 'log10'."
        )

    return pixel_to_value, calibration_description


# ============================================================
# 1. Validate the input
# ============================================================

validate_box(
    FIELD_BOX,
    image_width,
    image_height,
    "FIELD_BOX"
)

validate_box(
    COLORBAR_BOX,
    image_width,
    image_height,
    "COLORBAR_BOX"
)

orientation = COLORBAR_ORIENTATION.lower()

if orientation not in ("vertical", "horizontal"):
    raise ValueError(
        "COLORBAR_ORIENTATION must be 'vertical' or 'horizontal'."
    )


# ============================================================
# 2. Calibrate the colorbar
# ============================================================

pixel_to_value, calibration_description = calibrate_colorbar(
    TICK_PIXELS,
    TICK_VALUES,
    COLORBAR_SCALE
)

print("Colorbar calibration:")
print(calibration_description)


# ============================================================
# 3. Extract the colorbar
# ============================================================

cb_x_left, cb_x_right, cb_y_top, cb_y_bottom = COLORBAR_BOX

colorbar_image = rgb[
    cb_y_top:cb_y_bottom + 1,
    cb_x_left:cb_x_right + 1,
    :
].astype(float)

if orientation == "vertical":

    # Median across the colorbar width
    colorbar_colors = np.median(
        colorbar_image,
        axis=1
    )

    colorbar_pixel_coordinates = np.arange(
        cb_y_top,
        cb_y_bottom + 1,
        dtype=float
    )

else:

    # Median across the colorbar height
    colorbar_colors = np.median(
        colorbar_image,
        axis=0
    )

    colorbar_pixel_coordinates = np.arange(
        cb_x_left,
        cb_x_right + 1,
        dtype=float
    )

colorbar_values = pixel_to_value(
    colorbar_pixel_coordinates
)


# ============================================================
# 4. Extract the colored field
# ============================================================

field_x_left, field_x_right, field_y_top, field_y_bottom = FIELD_BOX

field_rgb = rgb[
    field_y_top:field_y_bottom + 1,
    field_x_left:field_x_right + 1,
    :
].astype(float)

ny, nx, _ = field_rgb.shape

print()
print("Recovered raster dimensions:")
print("nx =", nx)
print("ny =", ny)


# ============================================================
# 5. Match each field pixel to the colorbar
# ============================================================

# Build a nearest-neighbor search tree in RGB space
color_tree = cKDTree(colorbar_colors)

field_pixels = field_rgb.reshape(-1, 3)

rgb_distance, nearest_color_index = color_tree.query(
    field_pixels,
    k=1
)

# Convert nearest colorbar position into a scalar value
Z_image_coordinates = colorbar_values[
    nearest_color_index
].reshape(ny, nx)

rgb_distance = rgb_distance.reshape(ny, nx)

# Exclude pixels that do not resemble the colorbar
invalid_pixels = rgb_distance > MAX_RGB_DISTANCE

Z_image_coordinates[invalid_pixels] = np.nan


# ============================================================
# 6. Construct physical x-y coordinates
# ============================================================

x_min, x_max = X_RANGE
y_min, y_max = Y_RANGE

x = np.linspace(x_min, x_max, nx)
y = np.linspace(y_min, y_max, ny)

# Image coordinates increase downward.
# Flip the scalar field so physical y increases upward.
Z = np.flipud(Z_image_coordinates)

X, Y = np.meshgrid(x, y)


# ============================================================
# 7. Construct a colormap from the extracted colorbar
# ============================================================

# Matplotlib colormaps proceed from low value to high value.
color_order = np.argsort(colorbar_values)

recovered_colormap = ListedColormap(
    np.clip(
        colorbar_colors[color_order] / 255.0,
        0.0,
        1.0
    )
)


# ============================================================
# 8. Report reconstruction quality
# ============================================================

valid_values = np.isfinite(Z)

if np.count_nonzero(valid_values) == 0:
    raise RuntimeError(
        "No valid field pixels were recovered. "
        "Check FIELD_BOX, COLORBAR_BOX, and MAX_RGB_DISTANCE."
    )

masked_fraction = 1.0 - np.mean(valid_values)

print()
print("Recovered field statistics:")
print(f"Minimum value       : {np.nanmin(Z):.12e}")
print(f"Maximum value       : {np.nanmax(Z):.12e}")
print(f"Mean value          : {np.nanmean(Z):.12e}")
print(f"Masked pixel fraction: {100.0 * masked_fraction:.3f}%")

print()
print("RGB matching statistics:")
print(
    "Median RGB distance:",
    np.nanmedian(rgb_distance)
)
print(
    "95th percentile RGB distance:",
    np.nanpercentile(rgb_distance, 95)
)


# ============================================================
# 9. Select contour levels
# ============================================================

if CONTOUR_RANGE.lower() == "field":

    contour_min = np.nanmin(Z)
    contour_max = np.nanmax(Z)

elif CONTOUR_RANGE.lower() == "colorbar":

    contour_min = np.nanmin(colorbar_values)
    contour_max = np.nanmax(colorbar_values)

else:
    raise ValueError(
        "CONTOUR_RANGE must be 'field' or 'colorbar'."
    )

if np.isclose(contour_min, contour_max):
    raise RuntimeError(
        "The recovered field is nearly constant, so contour "
        "levels cannot be constructed."
    )

contour_levels = np.linspace(
    contour_min,
    contour_max,
    NUMBER_OF_CONTOURS
)


# ============================================================
# 10. Plot the recovered contour field
# ============================================================

fig, ax = plt.subplots(figsize=(9, 7))

filled_contours = ax.contourf(
    X,
    Y,
    np.ma.masked_invalid(Z),
    levels=contour_levels,
    cmap=recovered_colormap,
    extend="both"
)

contour_lines = ax.contour(
    X,
    Y,
    np.ma.masked_invalid(Z),
    levels=contour_levels,
    colors="black",
    linewidths=0.25,
    alpha=0.6
)

colorbar = fig.colorbar(
    filled_contours,
    ax=ax
)

colorbar.set_label("Recovered field value")

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Recovered contour plot")
ax.set_aspect("equal")

plt.tight_layout()


# ============================================================
# 11. Save the recovered data
# ============================================================

base_name = os.path.splitext(
    os.path.basename(filename)
)[0]

output_directory = f"{base_name}_recovered"
os.makedirs(output_directory, exist_ok=True)

contour_filename = os.path.join(
    output_directory,
    f"{base_name}_contour.png"
)

npz_filename = os.path.join(
    output_directory,
    f"{base_name}_field.npz"
)

x_filename = os.path.join(
    output_directory,
    f"{base_name}_x.csv"
)

y_filename = os.path.join(
    output_directory,
    f"{base_name}_y.csv"
)

z_filename = os.path.join(
    output_directory,
    f"{base_name}_Z.csv"
)

plt.savefig(
    contour_filename,
    dpi=300,
    bbox_inches="tight"
)

np.savez_compressed(
    npz_filename,
    x=x,
    y=y,
    Z=Z
)

np.savetxt(
    x_filename,
    x,
    delimiter=",",
    fmt="%.12e"
)

np.savetxt(
    y_filename,
    y,
    delimiter=",",
    fmt="%.12e"
)

np.savetxt(
    z_filename,
    Z,
    delimiter=",",
    fmt="%.12e"
)


# Optional long-format CSV: x, y, value
if SAVE_LONG_FORMAT_CSV:

    long_csv_filename = os.path.join(
        output_directory,
        f"{base_name}_xyz.csv"
    )

    xyz_data = np.column_stack([
        X.ravel(),
        Y.ravel(),
        Z.ravel()
    ])

    np.savetxt(
        long_csv_filename,
        xyz_data,
        delimiter=",",
        header="x,y,value",
        comments="",
        fmt="%.12e"
    )

plt.show()


# ============================================================
# 12. Zip and download all results
# ============================================================

zip_path = shutil.make_archive(
    output_directory,
    "zip",
    output_directory
)

print()
print("Saved output archive:", zip_path)

files.download(zip_path)